# [Feature Update (2026-07-14)] CREATE OR ALTER is generally available for 24 object types — 検証ノートブック

## このノートブックについて

Zenn 記事「[[Feature Update (2026-07-14)] CREATE OR ALTER is generally available for 24 object types](https://zenn.dev/gtk0326/articles/i165-feature-update-2026-07-14-create-or-alter-is-)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## セットアップ

検証用の DB・スキーマを作成します。


In [ ]:
USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS COA_DEMO;
CREATE SCHEMA IF NOT EXISTS COA_DEMO.PUBLIC;
USE SCHEMA COA_DEMO.PUBLIC;
USE WAREHOUSE COMPUTE_WH;

## ステップ1: TABLE を作成し、再実行でカラムを追加する

まず 2 カラムのテーブルを作成してデータを入れ、末尾に `email` を足した同じ文を再実行します。
既存の 2 行を保持したままカラムだけが追加されることを確認します。


In [ ]:
CREATE OR ALTER TABLE customers (
  id   NUMBER,
  name STRING
);

INSERT INTO customers VALUES (1, 'Alice'), (2, 'Bob');

CREATE OR ALTER TABLE customers (
  id    NUMBER,
  name  STRING,
  email STRING
);

SELECT * FROM customers ORDER BY id;

## ステップ2: WAREHOUSE のサイズを再実行で変更する

アカウントレベルのオブジェクトも同じパターンで管理できます。
X-Small / 60秒 で作成したウェアハウスが、再実行で Small / 120秒 に変わることを確認します。


In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR ALTER WAREHOUSE COA_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND   = 60
  AUTO_RESUME    = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE OR ALTER WAREHOUSE COA_WH
  WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND   = 120
  AUTO_RESUME    = TRUE
  INITIALLY_SUSPENDED = TRUE;

SHOW WAREHOUSES LIKE 'COA_WH';
SELECT "name", "size", "auto_suspend" FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

## 注意ポイント1: カラムリストの途中への追加はエラーで止まる

`name` を `customer_name` に変えた定義を再実行します。
Snowflake からは「途中位置への新カラム追加」に見えるため、**このセルは意図的にエラーになります。**

```
000002 (0A000): Unsupported feature
'CREATE OR ALTER TABLE column add before end of column list'.
```


In [ ]:
USE ROLE SYSADMIN;
USE SCHEMA COA_DEMO.PUBLIC;

-- このセルは意図的にエラーになります
CREATE OR ALTER TABLE customers (
  id            NUMBER,
  customer_name STRING,
  email         STRING
);

## 注意ポイント2: 末尾カラムの「リネーム」はエラーにならず、データが消える

`email` にデータを入れた状態で、末尾カラムを `mail_address` に変えた定義を再実行します。
ステートメントは**正常終了**しますが、旧カラムの DROP + 新カラムの ADD として実行されるため、メールアドレスのデータはすべて失われます。


In [ ]:
UPDATE customers SET email = CASE id WHEN 1 THEN 'alice@example.com' ELSE 'bob@example.com' END;

-- email にデータが入っていることを確認
SELECT * FROM customers ORDER BY id;

CREATE OR ALTER TABLE customers (
  id           NUMBER,
  name         STRING,
  mail_address STRING
);

-- mail_address はすべて NULL（データ消失）
SELECT * FROM customers ORDER BY id;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** ウェアハウスや DB を残すと意図しないクレジット消費につながります。


In [ ]:
USE ROLE ACCOUNTADMIN;
DROP WAREHOUSE IF EXISTS COA_WH;
USE ROLE SYSADMIN;
DROP DATABASE IF EXISTS COA_DEMO;